In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/"

print("Đang load data...")
log_main  = pd.read_csv(DATA_DIR + "log_standard_4_22_to_5_08_1k.csv")
log_prev  = pd.read_csv(DATA_DIR + "log_standard_4_08_to_4_21_1k.csv")
log_rand  = pd.read_csv(DATA_DIR + "log_random_4_22_to_5_08_1k.csv")
user_feat = pd.read_csv(DATA_DIR + "user_features_1k.csv")
vid_basic = pd.read_csv(DATA_DIR + "video_features_basic_1k.csv")

print(f" log_main  : {len(log_main):>10,} rows")
print(f" log_prev  : {len(log_prev):>10,} rows")
print(f" log_rand  : {len(log_rand):>10,} rows")
print(f" user_feat : {len(user_feat):>10,} rows")
print(f" vid_basic : {len(vid_basic):>10,} rows")

Đang load data...
 log_main  :  6,657,061 rows
 log_prev  :  5,055,984 rows
 log_rand  :     43,028 rows
 user_feat :      1,000 rows
 vid_basic :  4,371,868 rows


In [2]:
print("=== Làm sạch log_main ===")
print(f"Trước : {len(log_main):,} rows")

# 1. Xóa duplicate
log_main = log_main.drop_duplicates()
print(f"Sau drop_dup : {len(log_main):,} rows (xóa {59662:,} dòng)")

# 2. Tính watch_ratio — clip về [0,1], duration=0 → NaN
log_main['watch_ratio'] = (
    log_main['play_time_ms'] / log_main['duration_ms'].replace(0, np.nan)
).clip(0, 1)

# 3. Chuyển đổi thời gian
log_main['event_time'] = pd.to_datetime(log_main['time_ms'], unit='ms')
log_main['watch_time'] = log_main['play_time_ms'] / 1000  # ms → giây
log_main['duration']   = log_main['duration_ms']  / 1000  # ms → giây
log_main['is_share']   = log_main['is_forward']           # forward ≈ share

# 4. Tạo df_clean theo schema nhóm
SCHEMA_COLS = ['event_time', 'user_id', 'video_id',
               'watch_time', 'duration', 'watch_ratio',
               'is_click', 'is_like', 'is_share']
df_clean = log_main[SCHEMA_COLS].copy()

print(f"\nKết quả:")
print(f"  Rows          : {len(df_clean):,}")
print(f"  watch_ratio null (duration=0) : {df_clean['watch_ratio'].isnull().sum():,}")
print(f"\nDtypes:")
print(df_clean.dtypes)
print(f"\n5 dòng đầu:")
df_clean.head()

=== Làm sạch log_main ===
Trước : 6,657,061 rows
Sau drop_dup : 6,597,399 rows (xóa 59,662 dòng)

Kết quả:
  Rows          : 6,597,399
  watch_ratio null (duration=0) : 525,657

Dtypes:
event_time     datetime64[ms]
user_id                 int64
video_id                int64
watch_time            float64
duration              float64
watch_ratio           float64
is_click                int64
is_like                 int64
is_share                int64
dtype: object

5 dòng đầu:


,event_time,user_id,video_id,watch_time,duration,watch_ratio,is_click,is_like,is_share
0,2022-04-21 21:15:15.200,0,554434,3.925,7.833,0.501085,0,0,0
1,2022-04-21 21:15:27.466,0,1848093,0.000,106.900,0.000000,0,0,0
2,2022-04-21 21:15:27.466,0,4226937,0.000,130.533,0.000000,0,0,0
3,2022-04-21 21:15:27.466,0,553206,0.000,300.038,0.000000,0,0,0
4,2022-04-21 21:15:27.466,0,979987,0.000,37.463,0.000000,0,0,0


In [3]:
print("=== Làm sạch log_prev ===")
print(f"Trước : {len(log_prev):,} rows")

log_prev = log_prev.drop_duplicates()
log_prev['watch_ratio'] = (
    log_prev['play_time_ms'] / log_prev['duration_ms'].replace(0, np.nan)
).clip(0, 1)
log_prev['event_time'] = pd.to_datetime(log_prev['time_ms'], unit='ms')
log_prev['watch_time'] = log_prev['play_time_ms'] / 1000
log_prev['duration']   = log_prev['duration_ms']  / 1000
log_prev['is_share']   = log_prev['is_forward']

df_prev_clean = log_prev[SCHEMA_COLS].copy()
print(f"Sau   : {len(df_prev_clean):,} rows")
print(f"log_prev sạch xong")

=== Làm sạch log_prev ===
Trước : 5,055,984 rows
Sau   : 5,021,422 rows
log_prev sạch xong


In [4]:

print("=== Làm sạch user_features ===")
print(f"Nulls trước:\n{user_feat.isnull().sum()[user_feat.isnull().sum()>0]}")

# Fill null các cột số bằng median
num_cols = user_feat.select_dtypes(include='number').columns
user_feat[num_cols] = user_feat[num_cols].fillna(user_feat[num_cols].median())

# Fill null cột string bằng 'UNKNOWN'
str_cols = user_feat.select_dtypes(include='object').columns
user_feat[str_cols] = user_feat[str_cols].fillna('UNKNOWN')

# Đánh dấu low_active để Người 3 dùng lọc khi tính hot_score
user_feat['is_low_quality'] = user_feat['user_active_degree'].isin(
    ['low_active', 'single_low_active']
).astype(int)

print(f"\nNulls sau  : {user_feat.isnull().sum().sum()}")
print(f"Low quality users : {user_feat['is_low_quality'].sum()}")
print(f"user_features sạch xong")

=== Làm sạch user_features ===
Nulls trước:
onehot_feat4     33
onehot_feat12    33
onehot_feat13    33
onehot_feat14    33
onehot_feat15    33
onehot_feat16    33
onehot_feat17    33
dtype: int64

Nulls sau  : 0
Low quality users : 19
user_features sạch xong


/tmp/ipykernel_5822/526985620.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = user_feat.select_dtypes(include='object').columns


In [5]:
print("=== Làm sạch video_features_basic ===")
print(f"Trước : {len(vid_basic):,} rows")

# 1. Lọc bỏ video AD và UNKNOWN
vid_basic = vid_basic[vid_basic['video_type'] == 'NORMAL'].copy()
print(f"Sau lọc AD/UNKNOWN : {len(vid_basic):,} rows")

# 2. Chuyển duration ms → giây
vid_basic['video_duration_sec'] = vid_basic['video_duration'] / 1000

# 3. Fill null tag bằng 'unknown'
vid_basic['tag'] = vid_basic['tag'].fillna('unknown')

# 4. Fill null upload_dt
vid_basic['upload_dt'] = vid_basic['upload_dt'].fillna('unknown')

# 5. Drop các cột không cần thiết cho pipeline
DROP_COLS = ['server_width', 'server_height', 'music_id', 'music_type']
vid_basic = vid_basic.drop(columns=DROP_COLS)

print(f"Nulls còn lại:")
print(vid_basic.isnull().sum()[vid_basic.isnull().sum()>0])
print(f"\nColumns giữ lại: {list(vid_basic.columns)}")
print(f"video_features_basic sạch xong")

=== Làm sạch video_features_basic ===
Trước : 4,371,868 rows
Sau lọc AD/UNKNOWN : 4,361,638 rows
Nulls còn lại:
video_duration        572978
video_duration_sec    572978
dtype: int64

Columns giữ lại: ['video_id', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'tag', 'video_duration_sec']
video_features_basic sạch xong


In [6]:
print("=" * 50)
print("TỔNG KẾT LÀM SẠCH DATA")
print("=" * 50)
print(f"df_clean (log_main)  : {len(df_clean):>10,} rows  → pipeline chính")
print(f"df_prev_clean        : {len(df_prev_clean):>10,} rows  → train ML (Người 3)")
print(f"user_feat            : {len(user_feat):>10,} rows  → join khi cần")
print(f"vid_basic            : {len(vid_basic):>10,} rows  → join video info")
print()
print("Schema df_clean:")
print(df_clean.dtypes)
print()
print("Sample df_clean:")
df_clean.head(3)

TỔNG KẾT LÀM SẠCH DATA
df_clean (log_main)  :  6,597,399 rows  → pipeline chính
df_prev_clean        :  5,021,422 rows  → train ML (Người 3)
user_feat            :      1,000 rows  → join khi cần
vid_basic            :  4,361,638 rows  → join video info

Schema df_clean:
event_time     datetime64[ms]
user_id                 int64
video_id                int64
watch_time            float64
duration              float64
watch_ratio           float64
is_click                int64
is_like                 int64
is_share                int64
dtype: object

Sample df_clean:


,event_time,user_id,video_id,watch_time,duration,watch_ratio,is_click,is_like,is_share
0,2022-04-21 21:15:15.200,0,554434,3.925,7.833,0.501085,0,0,0
1,2022-04-21 21:15:27.466,0,1848093,0.000,106.900,0.000000,0,0,0
2,2022-04-21 21:15:27.466,0,4226937,0.000,130.533,0.000000,0,0,0


In [7]:
import os

SAMPLE_DIR = "../data/sample/"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Schema JSON cho Kafka — chuyển event_time về string
df_export = df_clean.copy()
df_export['event_time'] = df_export['event_time'].astype(str)

# Tạo 3 mức sample
samples = {
    '100k'  : 100_000,

}

for name, n in samples.items():
    sample = df_export.sample(n=n, random_state=42)
    path   = SAMPLE_DIR + f"kuairand_logs_{name}.csv"
    sample.to_csv(path, index=False)
    size   = os.path.getsize(path) / 1e6
    print(f" {path}  →  {n:>9,} dòng  ({size:.1f} MB)")

# Export full clean log (dùng cho Kafka producer)
full_path = SAMPLE_DIR + "kuairand_logs_full_clean.csv"
df_export.to_csv(full_path, index=False)
size_full = os.path.getsize(full_path) / 1e6
print(f" {full_path}  →  {len(df_export):>9,} dòng  ({size_full:.1f} MB)")

print(f"\n Xong! Kiểm tra thư mục data/sample/")

 ../data/sample/kuairand_logs_100k.csv  →    100,000 dòng  (6.7 MB)
 ../data/sample/kuairand_logs_full_clean.csv  →  6,597,399 dòng  (440.3 MB)

 Xong! Kiểm tra thư mục data/sample/
